In [1]:
%pip install transformers datasets accelerate

  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached xxhash-3.6.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (13 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached multidict-6.7.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.3 kB)
  Using cached propcache-0.4.1-cp313-cp313-macosx_11_0_arm64

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("in_domain_train.csv")

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["acceptable"])

print(f"Train: {len(train_df)}, Val: {len(val_df)}")
print(f"Train class distribution:\n{train_df['acceptable'].value_counts()}")
print(f"Val class distribution:\n{val_df['acceptable'].value_counts()}")

Train: 6295, Val: 1574
Train class distribution:
acceptable
1    4691
0    1604
Name: count, dtype: int64
Val class distribution:
acceptable
1    1173
0     401
Name: count, dtype: int64


In [5]:
train_df = train_df[["sentence", "acceptable"]]
val_df = val_df[["sentence", "acceptable"]]

train_df.groupby("acceptable").sample(1)

,sentence,acceptable
4638,"В группу студентов, направленных на полевую пр...",0
3735,Он вылил лейку воды на клумбу.,1


In [6]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "val": Dataset.from_pandas(val_df, preserve_index=False),
})

print(dataset)

/Users/kornilovyv/Desktop/learning/otus/nlp_basic/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['sentence', 'acceptable'],
        num_rows: 6295
    })
    val: Dataset({
        features: ['sentence', 'acceptable'],
        num_rows: 1574
    })
})


In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "sberbank-ai/ruBert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=512)

dataset = dataset.map(tokenize, batched=True, remove_columns=["sentence"])
dataset = dataset.rename_column("acceptable", "labels")
dataset.set_format("torch")

print(dataset)

Map: 100%|██████████| 1574/1574 [00:00<00:00, 103193.97 examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 6295
    })
    val: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1574
    })
})


In [29]:
import torch
from transformers import AutoModelForSequenceClassification

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)
print(f"Device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(device)

Device: mps


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 48115.90it/s]
BertForSequenceClassification LOAD REPORT from: sberbank-ai/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok 

In [30]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

BATCH_SIZE = 16

collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_dataloader = DataLoader(dataset["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=collator)
val_dataloader = DataLoader(dataset["val"], shuffle=False, batch_size=BATCH_SIZE, collate_fn=collator)

Оставляем для обучения только верхних четыре слоя - меньше времени на обучение, а самое главное более плавный выход на плато (меньше переобучение)

In [31]:
UNFREEZE_LAST_N = 4

for param in model.parameters():
    param.requires_grad = False

for layer in model.bert.encoder.layer[-UNFREEZE_LAST_N:]:
    for param in layer.parameters():
        param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Обучаемых параметров: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

Обучаемых параметров: 28,353,026 / 178,308,866 (15.9%)


In [32]:
import time
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

LR = 1e-5
MAX_EPOCHS = 8
PATIENCE = 2
WARMUP_RATIO = 0.1  # 10% шагов на warmup

total_steps = MAX_EPOCHS * len(train_dataloader)
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

best_val_loss = float("inf")
epochs_without_improvement = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss = 0
    t0 = time.time()

    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        train_loss += loss.item()

    train_loss /= len(train_dataloader)

    # --- Val ---
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in val_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            val_loss += outputs.loss.item()

    val_loss /= len(val_dataloader)
    elapsed = time.time() - t0

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "time": elapsed})
    print(f"Epoch {epoch} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | lr={scheduler.get_last_lr()[0]:.2e} | time={elapsed:.1f}s")

    # --- Plateau detection ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        model.save_pretrained("./rucola_classifier")
        tokenizer.save_pretrained("./rucola_classifier")
    else:
        epochs_without_improvement += 1
        print(f"  Val loss не улучшился ({epochs_without_improvement}/{PATIENCE})")
        if epochs_without_improvement >= PATIENCE:
            print("Плато достигнуто, останавливаем обучение.")
            break

print(f"\nЛучший val_loss: {best_val_loss:.4f}")

Epoch 1 | train_loss=0.5804 | val_loss=0.5532 | lr=9.72e-06 | time=29.0s


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.29it/s]


Epoch 2 | train_loss=0.5201 | val_loss=0.5133 | lr=8.33e-06 | time=28.6s


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.51it/s]


Epoch 3 | train_loss=0.4666 | val_loss=0.4956 | lr=6.94e-06 | time=28.8s


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.71it/s]


Epoch 4 | train_loss=0.4130 | val_loss=0.4967 | lr=5.56e-06 | time=29.2s
  Val loss не улучшился (1/2)
Epoch 5 | train_loss=0.3586 | val_loss=0.4983 | lr=4.17e-06 | time=28.9s
  Val loss не улучшился (2/2)
Плато достигнуто, останавливаем обучение.

Лучший val_loss: 0.4956


## Анализ результатов обучения

### Конфигурация экспериментов
 - Модель: sberbank-ai/ruBert-base
 - Датасет: RuCoLA (бинарная классификация грамматической приемлемости)
 - Train: 6295 примеров, Val: 1574 примера
 - Устройство: MPS (Apple Silicon)

---

### Эксперимент 1 — базовый (все слои обучаются)
lr=2e-5, weight_decay=0, без scheduler

| Эпоха | Train loss | Val loss | Время |
|-------|-----------|---------|-------|
| 1     | 0.5258    | **0.4692** | 92.8s |
| 2     | 0.3447    | 0.5455  | 83.4s |
| 3     | 0.1705    | 0.5807  | 83.7s |

Переобучение с первой эпохи, плато на эпохе 2. Время эпохи ~83-93с.

---

### Эксперимент 2 — заморозка нижних слоёв (UNFREEZE_LAST_N=2)
lr=2e-5, weight_decay=0.01, linear scheduler с warmup, 15.9% обучаемых параметров

| Эпоха | Train loss | Val loss | LR | Время |
|-------|-----------|---------|------|-------|
| 1     | 0.5925    | 0.5382  | 1.94e-05 | 22.2s |
| 2     | 0.5291    | 0.5266  | 1.67e-05 | 21.9s |
| 3     | 0.4802    | 0.4943  | 1.39e-05 | 21.6s |
| 4     | 0.4314    | 0.4928  | 1.11e-05 | 21.8s |
| 5     | 0.3804    | **0.4923** | 8.33e-06 | 21.7s |
| 6     | 0.3461    | 0.5162  | 5.56e-06 | 21.9s |

Модель стабильно улучшалась 5 эпох. Время эпохи сократилось до ~22с за счёт заморозки.

---

### Эксперимент 3 — больше незамороженных слоёв (UNFREEZE_LAST_N=4)
lr=2e-5, weight_decay=0.01, linear scheduler с warmup

| Эпоха | Train loss | Val loss | LR | Время |
|-------|-----------|---------|------|-------|
| 1     | 0.5692    | 0.5301  | 1.94e-05 | 30.2s |
| 2     | 0.4783    | **0.4672** | 1.67e-05 | 29.2s |
| 3     | 0.3811    | 0.4791  | 1.39e-05 | 28.9s |
| 4     | 0.2752    | 0.5613  | 1.11e-05 | 28.9s |

Лучший val_loss среди всех экспериментов (0.4672), но нестабильно — резкий рост после эпохи 2.

---

### Эксперимент 4 — UNFREEZE_LAST_N=4, меньший lr
lr=1e-5, weight_decay=0.01, linear scheduler с warmup

| Эпоха | Train loss | Val loss | LR | Время |
|-------|-----------|---------|------|-------|
| 1     | 0.5804    | 0.5532  | 9.72e-06 | 29.0s |
| 2     | 0.5201    | 0.5133  | 8.33e-06 | 28.6s |
| 3     | 0.4666    | **0.4956** | 6.94e-06 | 28.8s |
| 4     | 0.4130    | 0.4967  | 5.56e-06 | 29.2s |

Наиболее стабильное обучение: разрыв train/val минимален, val loss снижается плавно.

---

### Выводы
 - Заморозка нижних слоёв резко сокращает время эпохи (93с → 22-30с) и уменьшает переобучение
 - Лучший val_loss достигнут в эксп. 3 (0.4672), но нестабильно
 - Наиболее надёжный результат — эксп. 4: стабильное снижение val loss, малый разрыв с train
 - Оптимальное число эпох: 3-5 в зависимости от конфигурации

In [33]:
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, classification_report

# Загружаем тестовые данные
test_df = pd.read_csv("in_domain_dev.csv")[["sentence", "acceptable"]]
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_dataset = test_dataset.map(tokenize, batched=True, remove_columns=["sentence"])
test_dataset = test_dataset.rename_column("acceptable", "labels")
test_dataset.set_format("torch")

test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=BATCH_SIZE, collate_fn=collator)

# Загружаем лучшую сохранённую модель
best_model = AutoModelForSequenceClassification.from_pretrained("./rucola_classifier")
best_model = best_model.to(device)
best_model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_dataloader:
        labels = batch.pop("labels")
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = best_model(**batch).logits
        preds = logits.argmax(dim=-1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=["unacceptable", "acceptable"]))
print(f"Accuracy : {accuracy_score(all_labels, all_preds):.4f}")
print(f"F1 macro : {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"MCC      : {matthews_corrcoef(all_labels, all_preds):.4f}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9216.84it/s]


              precision    recall  f1-score   support

unacceptable       0.78      0.24      0.36       250
  acceptable       0.79      0.98      0.87       733

    accuracy                           0.79       983
   macro avg       0.78      0.61      0.62       983
weighted avg       0.79      0.79      0.74       983

Accuracy : 0.7884
F1 macro : 0.6176
MCC      : 0.3470


## Выбор метрики

Датасет RuCoLA несбалансирован: ~75% примеров — грамматически правильные (`acceptable=1`), ~25% — неправильные.

В этом случае **Accuracy** вводит в заблуждение: модель, которая всегда предсказывает `acceptable`, получит ~75% accuracy ничему не научившись.

**F1 macro** усредняет F1 по двум классам с равным весом — лучше, но не учитывает истинно отрицательные примеры.

**MCC (Matthews Correlation Coefficient)** — наиболее показательная метрика для несбалансированной бинарной классификации:
- Учитывает все четыре значения confusion matrix (TP, TN, FP, FN)
- Принимает значения от -1 до +1 (0 = случайные предсказания, 1 = идеально)

## Анализ результатов

| Метрика   | Значение |
|-----------|---------|
| Accuracy  | 0.7884  |
| F1 macro  | 0.6176  |
| **MCC**   | **0.3470** |

Accuracy (79%) выглядит неплохо, но вводит в заблуждение из-за дисбаланса классов.
Это подтверждает classification report:
- Класс `acceptable`: recall=0.98 — модель почти всегда правильно определяет грамматичные предложения
- Класс `unacceptable`: recall=0.24 — модель **пропускает 76% неграмматичных предложений**

MCC=0.35 — умеренный результат, заметно лучше случайного (0), но до хорошего качества (>0.6) далеко.
Модель смещена в сторону большинства: она научилась распознавать правильные предложения,
но плохо справляется с ошибочными.

**Вероятная причина:** дисбаланс классов (74% acceptable) в обучающей выборке.

In [34]:
## Анализ ошибок

errors_df = test_df.copy()
errors_df["predicted"] = all_preds
errors_df["true"] = all_labels
errors_df = errors_df[errors_df["predicted"] != errors_df["true"]]

errors_df["error_type"] = errors_df.apply(
    lambda r: "FP (предсказан acceptable, на самом деле нет)" if r["predicted"] == 1
    else "FN (предсказан unacceptable, на самом деле acceptable)", axis=1
)

pd.set_option("display.max_colwidth", 120)
errors_df[["sentence", "true", "predicted", "error_type"]].sample(10, random_state=42)

,sentence,true,predicted,error_type
865,Каждый день он приносил какую-нибудь охапку хвороста.,0,1,"FP (предсказан acceptable, на самом деле нет)"
81,Он промахнулся кулаком мне в лицо.,0,1,"FP (предсказан acceptable, на самом деле нет)"
423,"Тогда так зачем вообще венчаться, если не веришь?",0,1,"FP (предсказан acceptable, на самом деле нет)"
522,Я воспринял его как военного.,0,1,"FP (предсказан acceptable, на самом деле нет)"
875,Я долго питаю к ней нежные чувства.,0,1,"FP (предсказан acceptable, на самом деле нет)"
42,"На конференцию, которая проходила в Москве, прибыли триста один участник из всех регионов России.",0,1,"FP (предсказан acceptable, на самом деле нет)"
555,"Он стал торопливо размышлять, как бы более точнее ответить священнику и при этом не обидеть его.",0,1,"FP (предсказан acceptable, на самом деле нет)"
739,"Если Иван, к сожалению, в отпуске, вам придется ждать до сентября.",0,1,"FP (предсказан acceptable, на самом деле нет)"
93,"Только если операцию сделают немедленно, то останется еще какая-то надежда его спасти.",0,1,"FP (предсказан acceptable, на самом деле нет)"
804,Но тут вдали и вверху замигал огонек какой-то лампадки.,0,1,"FP (предсказан acceptable, на самом деле нет)"


## Обучение более простой модели cointegrated/rubert-tiny2

In [36]:
# cointegrated/rubert-tiny2 — значительно меньше ruBert-base (3 энкодер-слоя против 12)
# Меняем модель и токенизатор, пересоздаём датасет

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

MODEL_NAME = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=512)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "val": Dataset.from_pandas(val_df, preserve_index=False),
})
dataset = dataset.map(tokenize, batched=True, remove_columns=["sentence"])
dataset = dataset.rename_column("acceptable", "labels")
dataset.set_format("torch")

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_dataloader = DataLoader(dataset["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=collator)
val_dataloader = DataLoader(dataset["val"], shuffle=False, batch_size=BATCH_SIZE, collate_fn=collator)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(device)

print(dataset)
print(f"\nDevice: {device}")

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 22634.10it/s]
BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 6295
    })
    val: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1574
    })
})

Device: mps


In [37]:
import time
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

LR = 1e-5
MAX_EPOCHS = 8
PATIENCE = 2
WARMUP_RATIO = 0.1

total_steps = MAX_EPOCHS * len(train_dataloader)
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

best_val_loss = float("inf")
epochs_without_improvement = 0
history_tiny = []

for epoch in range(1, MAX_EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss = 0
    t0 = time.time()

    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        train_loss += loss.item()

    train_loss /= len(train_dataloader)

    # --- Val ---
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in val_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            val_loss += outputs.loss.item()

    val_loss /= len(val_dataloader)
    elapsed = time.time() - t0

    history_tiny.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "time": elapsed})
    print(f"Epoch {epoch} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | lr={scheduler.get_last_lr()[0]:.2e} | time={elapsed:.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        model.save_pretrained("./rucola_tiny2")
        tokenizer.save_pretrained("./rucola_tiny2")
    else:
        epochs_without_improvement += 1
        print(f"  Val loss не улучшился ({epochs_without_improvement}/{PATIENCE})")
        if epochs_without_improvement >= PATIENCE:
            print("Плато достигнуто, останавливаем обучение.")
            break

print(f"\nЛучший val_loss: {best_val_loss:.4f}")

Epoch 1 | train_loss=0.6044 | val_loss=0.5481 | lr=9.72e-06 | time=21.0s


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 14.95it/s]


Epoch 2 | train_loss=0.5419 | val_loss=0.5362 | lr=8.33e-06 | time=15.1s


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 17.77it/s]


Epoch 3 | train_loss=0.5264 | val_loss=0.5353 | lr=6.94e-06 | time=14.9s


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 19.29it/s]


Epoch 4 | train_loss=0.5112 | val_loss=0.5290 | lr=5.56e-06 | time=14.5s


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 19.58it/s]


Epoch 5 | train_loss=0.4952 | val_loss=0.5296 | lr=4.17e-06 | time=14.5s
  Val loss не улучшился (1/2)
Epoch 6 | train_loss=0.4849 | val_loss=0.5295 | lr=2.78e-06 | time=14.7s
  Val loss не улучшился (2/2)
Плато достигнуто, останавливаем обучение.

Лучший val_loss: 0.5290


In [39]:
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, classification_report

# Загружаем тестовые данные с токенизатором tiny2
test_df = pd.read_csv("in_domain_dev.csv")[["sentence", "acceptable"]]
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_dataset = test_dataset.map(tokenize, batched=True, remove_columns=["sentence"])
test_dataset = test_dataset.rename_column("acceptable", "labels")
test_dataset.set_format("torch")

test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=BATCH_SIZE, collate_fn=collator)

# Загружаем лучшую сохранённую модель tiny2
best_tiny = AutoModelForSequenceClassification.from_pretrained("./rucola_tiny2")
best_tiny = best_tiny.to(device)
best_tiny.eval()

tiny_preds, tiny_labels = [], []

with torch.no_grad():
    for batch in test_dataloader:
        labels = batch.pop("labels")
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = best_tiny(**batch).logits
        preds = logits.argmax(dim=-1).cpu()
        tiny_preds.extend(preds.tolist())
        tiny_labels.extend(labels.tolist())

print("=== rubert-tiny2 ===")
print(classification_report(tiny_labels, tiny_preds, target_names=["unacceptable", "acceptable"]))
print(f"Accuracy : {accuracy_score(tiny_labels, tiny_preds):.4f}")
print(f"F1 macro : {f1_score(tiny_labels, tiny_preds, average='macro'):.4f}")
print(f"MCC      : {matthews_corrcoef(tiny_labels, tiny_preds):.4f}")

Loading weights: 100%|██████████| 57/57 [00:00<00:00, 9432.84it/s]


=== rubert-tiny2 ===
              precision    recall  f1-score   support

unacceptable       0.66      0.20      0.31       250
  acceptable       0.78      0.96      0.86       733

    accuracy                           0.77       983
   macro avg       0.72      0.58      0.59       983
weighted avg       0.75      0.77      0.72       983

Accuracy : 0.7711
F1 macro : 0.5873
MCC      : 0.2731


## Сравнение моделей: ruBert-base vs rubert-tiny2

### Метрики на тестовой выборке (in_domain_dev.csv)

| Метрика | ruBert-base | rubert-tiny2 |
|---------|------------|--------------|
| Accuracy | 0.7884 | 0.7711 |
| F1 macro | 0.6176 | 0.5873 |
| **MCC** | **0.3470** | **0.2731** |

### По классам

| | ruBert-base | | rubert-tiny2 | |
|---|---|---|---|---|
| Класс | Precision | Recall | Precision | Recall |
| unacceptable | 0.78 | 0.24 | 0.66 | 0.20 |
| acceptable | 0.79 | 0.98 | 0.78 | 0.96 |

### Характеристики обучения

| | ruBert-base | rubert-tiny2 |
|---|---|---|
| Параметров | ~178M | ~29M |
| Время эпохи | ~29с | ~15с |
| Лучший val_loss | 0.4956 | 0.5290 |

### Выводы

Обе модели демонстрируют одинаковую проблему: **низкий recall на классе `unacceptable`**
(0.24 и 0.20 соответственно) из-за дисбаланса классов в датасете (~75% acceptable).

**ruBert-base** превосходит tiny2 по всем метрикам:
- MCC выше на 0.074 (0.347 vs 0.273)
- F1 macro выше на 0.030
- Лучше распознаёт неграмматичные предложения (recall 0.24 vs 0.20)

**rubert-tiny2** в 2 раза быстрее обучается и почти не переобучается,
но уступает в качестве из-за меньшей ёмкости модели.

**Главный вывод:** для задачи грамматической приемлемости ruBert-base предпочтительнее.
Ключевое направление улучшения для обеих моделей — борьба с дисбалансом классов.

## Влияние гиперпараметров на обучение

In [50]:
import gc
import time
import itertools
import numpy as np
from sklearn.metrics import f1_score, matthews_corrcoef
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
from datasets import Dataset, DatasetDict

# очищаем память от предыдущих моделей
for var in ["model", "best_model", "best_tiny"]:
    if var in dir():
        del globals()[var]
torch.mps.empty_cache()
gc.collect()

BASE_MODEL = "sberbank-ai/ruBert-base"
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def base_tokenize(batch):
    return base_tokenizer(batch["sentence"], truncation=True, max_length=512)

base_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "val":   Dataset.from_pandas(val_df,   preserve_index=False),
})
base_dataset = base_dataset.map(base_tokenize, batched=True, remove_columns=["sentence"])
base_dataset = base_dataset.rename_column("acceptable", "labels")

base_collator = DataCollatorWithPadding(tokenizer=base_tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1":  f1_score(labels, preds, average="macro"),
        "mcc": matthews_corrcoef(labels, preds),
    }

# сетка гиперпараметров (batch_size=32 не влезает в память MPS)
lr_values   = [1e-5, 2e-5, 5e-5]
batch_sizes = [16, 8]
results = []

for lr, batch_size in itertools.product(lr_values, batch_sizes):
    print(f"\n--- lr={lr:.0e}, batch_size={batch_size} ---")

    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)

    args = TrainingArguments(
        output_dir="./hp_search",
        num_train_epochs=4,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        logging_strategy="no",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=base_dataset["train"],
        eval_dataset=base_dataset["val"],
        data_collator=base_collator,
        compute_metrics=compute_metrics,
    )

    t0 = time.time()
    trainer.train()
    total_time = time.time() - t0

    eval_logs = [x for x in trainer.state.log_history if "eval_f1" in x]
    best = max(eval_logs, key=lambda x: x["eval_f1"])
    best_f1  = round(best["eval_f1"],  4)
    best_mcc = round(best["eval_mcc"], 4)
    epochs_run = len(eval_logs)
    time_per_epoch = round(total_time / epochs_run)
    print(f"  F1={best_f1} | MCC={best_mcc} | time={total_time:.0f}s ({time_per_epoch}s/epoch)")
    results.append({"lr": lr, "batch_size": batch_size, "epochs": epochs_run,
                    "val_f1": best_f1, "val_mcc": best_mcc,
                    "time_s": round(total_time), "time_per_epoch_s": time_per_epoch})

    del model, trainer
    torch.mps.empty_cache()
    gc.collect()

results_df = pd.DataFrame(results).sort_values("val_f1", ascending=False)
print("\n=== Итоговая таблица ===")
print(results_df.to_string(index=False))

Map: 100%|██████████| 1574/1574 [00:00<00:00, 74717.73 examples/s]



--- lr=1e-05, batch_size=16 ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 54340.27it/s]
BertForSequenceClassification LOAD REPORT from: sberbank-ai/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok 

Epoch,Training Loss,Validation Loss,F1,Mcc
1,No log,0.525665,0.511755,0.197691
2,No log,0.478753,0.679258,0.398730
3,No log,0.542407,0.693885,0.410684
4,No log,0.651522,0.677534,0.412330


/Users/kornilovyv/Desktop/learning/otus/nlp_basic/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/kornilovyv/Desktop/learning/otus/nlp_basic/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/kornilovyv/Desktop/learning/otus/nlp_basic/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

  F1=0.701 | MCC=0.4067 | time=295s (74s/epoch)

--- lr=5e-05, batch_size=8 ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 62325.75it/s]
BertForSequenceClassification LOAD REPORT from: sberbank-ai/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok 

Epoch,Training Loss,Validation Loss,F1,Mcc
1,No log,0.527479,0.615552,0.305905
2,No log,0.588580,0.654300,0.369701
3,No log,0.915873,0.687849,0.378101
4,No log,1.239491,0.684562,0.388413


/Users/kornilovyv/Desktop/learning/otus/nlp_basic/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/kornilovyv/Desktop/learning/otus/nlp_basic/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/kornilovyv/Desktop/learning/otus/nlp_basic/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  F1=0.6878 | MCC=0.3781 | time=460s (115s/epoch)

=== Итоговая таблица ===
     lr  batch_size  epochs  val_f1  val_mcc  time_s  time_per_epoch_s
0.00002           8       4  0.7152   0.4409     458               114
0.00002          16       4  0.7137   0.4343     292                73
0.00005          16       4  0.7010   0.4067     295                74
0.00001           8       4  0.6941   0.4178     463               116
0.00001          16       4  0.6939   0.4107     292                73
0.00005           8       4  0.6878   0.3781     460               115


## Выводы по влиянию гиперпараметров

### Результаты экспериментов (ruBert-base, 4 эпохи, warmup_ratio=0.1)

| lr | batch_size | val_f1 | val_mcc | time_per_epoch |
|----|-----------|--------|---------|----------------|
| **2e-05** | **8** | **0.7152** | **0.4409** | 114s |
| 2e-05 | 16 | 0.7137 | 0.4343 | 73s |
| 5e-05 | 16 | 0.7010 | 0.4067 | 74s |
| 1e-05 | 8  | 0.6941 | 0.4178 | 116s |
| 1e-05 | 16 | 0.6939 | 0.4107 | 73s |
| 5e-05 | 8  | 0.6878 | 0.3781 | 115s |

### Влияние learning rate

`lr=2e-5` — оптимальное значение: лучший F1 (0.715) и MCC (0.44) при обоих batch_size.

`lr=1e-5` — обучение слишком медленное: за 4 эпохи модель не успевает достаточно адаптироваться. При большем числе эпох результаты могли бы улучшиться.

`lr=5e-5` — слишком высокий: модель обновляется слишком резко, что приводит к нестабильности. При `batch_size=8` это особенно заметно — MCC падает до 0.378, худший результат среди всех.

**LR — наиболее важный гиперпараметр**: разрыв между лучшим (`2e-5`) и худшим (`5e-5` + `bs=8`) составляет 0.063 по F1 и 0.063 по MCC.

### Влияние batch_size

При `lr=2e-5` разница между `bs=8` и `bs=16` минимальна: +0.0015 F1 и +0.007 MCC в пользу меньшего батча, но `bs=8` в **1.6 раза медленнее** (114с vs 73с на эпоху).

При `lr=5e-5` картина другая: `bs=16` значительно лучше `bs=8` (0.407 vs 0.378 MCC) — большой батч стабилизирует градиенты при высоком lr.

**Вывод по batch_size:** при умеренном lr (`2e-5`) разница несущественна и `bs=16` предпочтительнее из-за скорости. При высоком lr (`5e-5`) больший батч помогает стабилизировать обучение.

### Оптимальная конфигурация

`lr=2e-5`, `batch_size=16` — лучший баланс качества и скорости:
- val_f1 = 0.7137, val_mcc = 0.4343
- Время эпохи: 73с (в 1.6 раза быстрее bs=8 при минимальной разнице в качестве)

### Сравнение с ручным обучением

Trainer (без заморозки слоёв) значительно превзошёл ручной цикл с заморозкой:
- Лучший MCC в ручных экспериментах: ~0.35
- Лучший MCC с Trainer: **0.44**

Ключевая причина: заморозка нижних слоёв ограничивала адаптацию модели. Полное дообучение всех слоёв при правильном lr даёт существенный прирост качества.

### MCC vs Loss

Trainer с eval_strategy="epoch" и выбором лучшей эпохи по F1/MCC постфактум — более гибкий подход, чем ручная ранняя остановка по val_loss. Он позволяет найти оптимум по нужной метрике, не прерывая обучение раньше времени.

Когда val loss > train loss — модель хуже откалибрована на валидации (вероятности менее точные). Но MCC смотрит только на финальное решение argmax, а не на вероятности. Поэтому возможна ситуация:

 - Модель стала более "решительной" — сдвигает предсказания в сторону правильного класса
 - argmax чаще попадает в нужный класс → MCC растёт
 - Но уверенность в вероятностях снизилась → val loss растёт

То есть модель правильно классифицирует, но с плохой калибровкой. Для задачи классификации (нам важен только класс, а не вероятность) — это допустимо. Для задач где важна вероятность (например, risk scoring) — это проблема.

## Эксперимент: борьба с дисбалансом классов

Лучшие гиперпараметры из поиска: `lr=2e-5`, `batch_size=16`.

Применяем два дополнительных улучшения:

1. **Взвешивание классов в loss** — штраф за ошибку на редком классе (`unacceptable`) пропорционально больше. Веса рассчитываются автоматически из частоты классов в train.

2. **Сохранение лучшей модели по MCC** — вместо val_loss используем целевую метрику напрямую, что соответствует задаче.

Ожидаем улучшения recall на классе `unacceptable` и рост MCC.

In [51]:
import gc
import numpy as np
import torch
from sklearn.metrics import f1_score, matthews_corrcoef
from transformers import (
    AutoModelForSequenceClassification, DataCollatorWithPadding,
    TrainingArguments, Trainer
)

# подсчёт весов классов по частоте в train
class_counts = train_df["acceptable"].value_counts().sort_index()
total = class_counts.sum()
class_weights = torch.tensor([total / class_counts[0], total / class_counts[1]], dtype=torch.float)
print(f"Веса классов: unacceptable={class_weights[0]:.2f}, acceptable={class_weights[1]:.2f}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weights = class_weights.to(model.device)
        loss = torch.nn.CrossEntropyLoss(weight=weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1":  f1_score(labels, preds, average="macro"),
        "mcc": matthews_corrcoef(labels, preds),
    }

torch.mps.empty_cache()
gc.collect()

model = AutoModelForSequenceClassification.from_pretrained("sberbank-ai/ruBert-base", num_labels=2)
collator = DataCollatorWithPadding(tokenizer=base_tokenizer)

args = TrainingArguments(
    output_dir="./rucola_weighted",
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mcc",
    greater_is_better=True,
    logging_strategy="epoch",
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=base_dataset["train"],
    eval_dataset=base_dataset["val"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./rucola_weighted_best")
base_tokenizer.save_pretrained("./rucola_weighted_best")

Веса классов: unacceptable=3.92, acceptable=1.34


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 57598.96it/s]
BertForSequenceClassification LOAD REPORT from: sberbank-ai/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok 

Epoch,Training Loss,Validation Loss,F1,Mcc
1,0.667448,0.622374,0.656512,0.319785
2,0.470676,0.660895,0.683186,0.369831
3,0.248869,0.797733,0.700789,0.401634
4,0.150746,1.516287,0.696089,0.409067
5,0.089215,1.919096,0.690262,0.400068
6,0.059113,2.080242,0.681314,0.389133


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]
/Users/kornilovyv/Desktop/learning/otus/nlp_basic/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.04it/s]
/Users/kornilovyv/Desktop/learning/otus/nlp_basic/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]
/Users/kornilovyv/Desktop/learning/otus/nlp_basic/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writ

('./rucola_weighted_best/tokenizer_config.json',
 './rucola_weighted_best/tokenizer.json')

In [52]:
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, classification_report
from torch.utils.data import DataLoader
from datasets import Dataset

# загружаем тестовую выборку с токенизатором ruBert-base
test_df = pd.read_csv("in_domain_dev.csv")[["sentence", "acceptable"]]
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_dataset = test_dataset.map(base_tokenize, batched=True, remove_columns=["sentence"])
test_dataset = test_dataset.rename_column("acceptable", "labels")
test_dataset.set_format("torch")

test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=16, collate_fn=collator)

# загружаем лучшую модель
best_weighted = AutoModelForSequenceClassification.from_pretrained("./rucola_weighted_best")
best_weighted = best_weighted.to(device)
best_weighted.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_dataloader:
        labels = batch.pop("labels")
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = best_weighted(**batch).logits.argmax(-1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=["unacceptable", "acceptable"]))
print(f"Accuracy : {accuracy_score(all_labels, all_preds):.4f}")
print(f"F1 macro : {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"MCC      : {matthews_corrcoef(all_labels, all_preds):.4f}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 23333.94it/s]


              precision    recall  f1-score   support

unacceptable       0.62      0.42      0.50       250
  acceptable       0.82      0.91      0.86       733

    accuracy                           0.79       983
   macro avg       0.72      0.67      0.68       983
weighted avg       0.77      0.79      0.77       983

Accuracy : 0.7874
F1 macro : 0.6830
MCC      : 0.3840


## Результаты: взвешенная модель vs базовая

Веса классов: `unacceptable=3.92`, `acceptable=1.34` — рассчитаны автоматически из частоты в train.

### Метрики на тестовой выборке

| Метрика | ruBert-base базовый | ruBert-base weighted |
|---------|---------------------|----------------------|
| Accuracy | 0.7884 | 0.7874 |
| F1 macro | 0.6176 | **0.6830** |
| MCC | 0.3470 | **0.3840** |

### По классам

| Класс | Precision (base) | Recall (base) | Precision (weighted) | Recall (weighted) |
|-------|-----------------|--------------|----------------------|-------------------|
| unacceptable | 0.78 | 0.24 | 0.62 | **0.42** |
| acceptable | 0.79 | 0.98 | 0.82 | 0.91 |

### Выводы

Взвешивание классов дало значительное улучшение по всем целевым метрикам:
- **Recall на `unacceptable` вырос с 0.24 до 0.42** (+75%) — модель стала находить почти вдвое больше неграмматичных предложений
- **F1 macro +0.066**, **MCC +0.037**

Получился компромисс: precision на `unacceptable` снизился (0.78 → 0.62) — модель стала чаще ошибочно отмечать грамматичные предложения как неграмматичные. Recall на `acceptable` незначительно снизился (0.98 → 0.91).

Accuracy практически не изменилась (0.7884 → 0.7874) — accuracy не является информативной метрикой при дисбалансе классов.